# Week 49, Day 3: Language as prediction: surprisal and processing difficulty

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w049/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Bigram surprisal

Write bigram_surprisal(sentence): for each word after the first, return (word, -log2 P(word | previous word)) using the smoothed bigram probability with alpha = 0.1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
corpus = """the cat sat on the mat . the dog sat on the rug . the cat chased the mouse . the dog chased the cat .
the child read the book . the child read the letter . the teacher read the book . the teacher wrote the letter .
the girl ate the apple . the boy ate the bread . the girl drank the milk . the boy drank the water .
the farmer milked the cow . the farmer fed the cow . the farmer fed the dog . the baker baked the bread .
the baker sold the bread . the man drove the car . the woman drove the bus . the man parked the car .
the cat slept on the mat . the dog slept on the rug . the child slept in the bed . the baby slept in the cot .
she spread butter on the bread . he spread jam on the bread . she put milk in the tea . he put sugar in the tea .""".split()
V = sorted(set(corpus))
uni = Counter(corpus[:-1]); big = Counter(zip(corpus[:-1], corpus[1:]))

def p_bigram(w, prev, alpha=0.1):
    return (big[(prev, w)] + alpha) / (uni[prev] + alpha * len(V))

def bigram_surprisal(sentence):
    ws = sentence.split()
    return ...

for s in ["the farmer milked the cow .", "the farmer milked the bread .", "she spread butter on the bread .", "she spread butter on the socks ."]:
    print(s, " ".join(f"{w}:{v:.1f}" for w, v in bigram_surprisal(s)))

In [ ]:
import numpy as np
def test_socks():
    "under the bigram model, 'socks' after 'butter on the' is at least 5 bits more surprising than 'bread'"
    s_bread = dict(bigram_surprisal("she spread butter on the bread ."))["bread"]
    s_socks = dict(bigram_surprisal("she spread butter on the socks ."))["socks"]
    assert s_socks > s_bread + 5
def test_blind():
    "the bigram model cannot prefer 'cow' over 'bread' after 'milked the'"
    s_cow = dict(bigram_surprisal("the farmer milked the cow ."))["cow"]
    s_bread = dict(bigram_surprisal("the farmer milked the bread ."))["bread"]
    assert s_cow >= s_bread - 0.5

test_socks()
test_blind()
print("All checks passed")

### Exercise: Two words of context

Write p_trigram(w, u, v) by interpolation (use the empirical trigram and bigram frequencies, 0 when the context was never seen, and an add-0.5 unigram estimate), and trigram_surprisal(sentence) for words from the third onwards.

In [ ]:
tri = Counter(zip(corpus[:-2], corpus[1:-1], corpus[2:])); bictx = Counter(zip(corpus[:-2], corpus[1:-1]))
U = Counter(corpus)

def p_trigram(w, u, v, lam=(0.6, 0.3, 0.1)):
    pt = tri[(u, v, w)] / bictx[(u, v)] if bictx[(u, v)] else 0.0
    pb = big[(v, w)] / uni[v] if uni[v] else 0.0
    pu = (U[w] + 0.5) / (len(corpus) + 0.5 * len(V))
    return ...

def trigram_surprisal(sentence):
    ws = sentence.split()
    return [(w, -np.log2(p_trigram(w, ws[i - 2], ws[i - 1]))) for i, w in enumerate(ws) if i > 1]

for s in ["the farmer milked the cow .", "the farmer milked the bread ."]:
    print(s, " ".join(f"{w}:{v:.1f}" for w, v in trigram_surprisal(s)))

In [ ]:
import numpy as np
def test_cow():
    "the trigram model finds 'cow' after 'milked the' at least 3 bits less surprising than 'bread'"
    s_cow = dict(trigram_surprisal("the farmer milked the cow ."))["cow"]
    s_bread = dict(trigram_surprisal("the farmer milked the bread ."))["bread"]
    assert s_bread > s_cow + 3

test_cow()
print("All checks passed")

### Exercise: Recovering the surprisal effect

Build a design matrix with an intercept, surprisal and length for every word of every sentence and reader, simulate reading times with noise of sd 40 ms, and fit by np.linalg.lstsq.

In [ ]:
sentences = ["the farmer milked the cow .", "the farmer milked the bread .", "she spread butter on the bread .", "she spread butter on the socks .",
             "the child read the book .", "the child read the milk .", "the dog chased the cat .", "the dog chased the tea .",
             "the man drove the car .", "the man drove the apple .", "he put sugar in the tea .", "he put sugar in the bus ."]
rows = []
for s in sentences:
    for w, sv in trigram_surprisal(s):
        rows.append((sv, len(w)))
rows = np.array(rows)
X_one = np.c_[np.ones(len(rows)), rows]
X = np.tile(X_one, (40, 1))
rt = ...
beta = ...
print(f"intercept {beta[0]:.0f} ms, surprisal {beta[1]:.1f} ms per bit, length {beta[2]:.1f} ms per letter")
plt.scatter(X[:, 1], rt, s=3, alpha=0.3); plt.xlabel("surprisal (bits)"); plt.ylabel("reading time (ms)"); plt.show()

In [ ]:
import numpy as np
def test_slope():
    "the estimated surprisal coefficient is within 3 ms per bit of 20"
    assert abs(beta[1] - 20) < 3
def test_length():
    "the length coefficient is within 3 ms per letter of 8"
    assert abs(beta[2] - 8) < 3

test_slope()
test_length()
print("All checks passed")